In [ ]:
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [ ]:
# A → B → C → D
k1_seq = 2.0    # A → B
k2_seq = 1.5    # B → C
k3_seq = 1.0    # C → D

# A → B → D и A → B → C → D
k1_hyb = 1.8    # A → B
k2_hyb = 0.7    # B → D
k3_hyb = 0.5    # B → C
k4_hyb = 0.9    # C → D

time_step = 0.1
time_steps = np.arange(0, 2.0, time_step)   # 0, 0.1, ..., 1.9
n_measurements = len(time_steps)
n_samples_per_reaction = 2000
total_samples = 2 * n_samples_per_reaction

def generate_initial_concentrations():
    A = np.random.uniform(10, 100)
    return A, 0.0, 0.0, 0.0

# Реакция 1: A → B → C → D 
def reaction1_system(t, A, B, C, D, k1, k2, k3):
    dA_dt = -k1 * A
    dB_dt = k1 * A - k2 * B
    dC_dt = k2 * B - k3 * C
    dD_dt = k3 * C
    return dA_dt, dB_dt, dC_dt, dD_dt

# Реакция 2: A → B → D и A → B → C → D 
def reaction2_system(t, A, B, C, D, k1, k2, k3, k4):
    dA_dt = -k1 * A
    dB_dt = k1 * A - k2 * B - k3 * B
    dC_dt = k3 * B - k4 * C
    dD_dt = k2 * B + k4 * C
    return dA_dt, dB_dt, dC_dt, dD_dt

def solve_ode(system_func, initial_conditions, rate_constants, time_steps):
    A0, B0, C0, D0 = initial_conditions
    concentrations = [(A0, B0, C0, D0)]

    for i in range(1, len(time_steps)):
        dt = time_steps[i] - time_steps[i-1]
        A_prev, B_prev, C_prev, D_prev = concentrations[-1]

        dA, dB, dC, dD = system_func(
            time_steps[i-1],
            A_prev, B_prev, C_prev, D_prev,
            *rate_constants
        )

        A_new = max(0, A_prev + dA * dt)
        B_new = max(0, B_prev + dB * dt)
        C_new = max(0, C_prev + dC * dt)
        D_new = max(0, D_prev + dD * dt)

        concentrations.append((A_new, B_new, C_new, D_new))

    return concentrations

data = []

# Реакция 1
for _ in range(n_samples_per_reaction):
    A0, B0, C0, D0 = generate_initial_concentrations()
    concentrations = solve_ode(
        reaction1_system,
        (A0, B0, C0, D0),
        (k1_seq, k2_seq, k3_seq),
        time_steps
    )

    noisy_A, noisy_B, noisy_C, noisy_D = [], [], [], []

    for A, B, C, D in concentrations:
        noise_A = np.random.uniform(-0.05, 0.05) * A
        noise_B = np.random.uniform(-0.05, 0.05) * B
        noise_C = np.random.uniform(-0.05, 0.05) * C
        noise_D = np.random.uniform(-0.05, 0.05) * D

        noisy_A.append(max(0, round(A + noise_A, 3)))
        noisy_B.append(max(0, round(B + noise_B, 3)))
        noisy_C.append(max(0, round(C + noise_C, 3)))
        noisy_D.append(max(0, round(D + noise_D, 3)))

    row = noisy_A + noisy_B + noisy_C + noisy_D + [1]  # класс 1
    data.append(row)

# Реакция 2
for _ in range(n_samples_per_reaction):
    A0, B0, C0, D0 = generate_initial_concentrations()
    concentrations = solve_ode(
        reaction2_system,
        (A0, B0, C0, D0),
        (k1_hyb, k2_hyb, k3_hyb, k4_hyb),
        time_steps
    )

    noisy_A, noisy_B, noisy_C, noisy_D = [], [], [], []

    for A, B, C, D in concentrations:
        noise_A = np.random.uniform(-0.05, 0.05) * A
        noise_B = np.random.uniform(-0.05, 0.05) * B
        noise_C = np.random.uniform(-0.05, 0.05) * C
        noise_D = np.random.uniform(-0.05, 0.05) * D

        noisy_A.append(max(0, round(A + noise_A, 3)))
        noisy_B.append(max(0, round(B + noise_B, 3)))
        noisy_C.append(max(0, round(C + noise_C, 3)))
        noisy_D.append(max(0, round(D + noise_D, 3)))

    row = noisy_A + noisy_B + noisy_C + noisy_D + [2]  # класс 2 
    data.append(row)

columns = []
for i in range(1, n_measurements + 1):
    columns.append(f'A{i}')
for i in range(1, n_measurements + 1):
    columns.append(f'B{i}')
for i in range(1, n_measurements + 1):
    columns.append(f'C{i}')
for i in range(1, n_measurements + 1):
    columns.append(f'D{i}')
columns.append('Reaction')

df = pd.DataFrame(data, columns=columns)
df.to_csv('data.csv', index=False)
df.to_excel('data.xlsx', index=False)
